In [4]:
!pip install numpy pandas scikit-learn torch torchvision torchaudio
!pip install pytorch-tabnet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 73.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.

In [5]:
# =========================================
# 📚 Imports
# =========================================
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pytorch_tabnet.tab_model import TabNetClassifier
import joblib
import os


In [6]:
# =========================================
# ⚙️ Configurations
# =========================================
CSV_PATH = "/kaggle/input/rewrtfyuiop/attack_data.csv"  # 👈 change this to your dataset path
TARGET_COL = "attack_cat_merged_labeled"
OUTPUT_DIR = "./models"

NOISE_DIM = 64
GAN_EPOCHS = 200
GAN_BATCH_SIZE = 1024
GAN_LR = 2e-4

MAX_SAMPLES_PER_CLASS = 5000

TABNET_EPOCHS = 100
TABNET_LR = 2e-2
TABNET_BATCH_SIZE = 1024
TABNET_VBATCH_SIZE = 128

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [7]:
# =========================================
# 🧩 Utility functions
# =========================================
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def to_onehot(labels, n_classes, device):
    labels = labels.reshape(-1)
    oh = torch.zeros((labels.shape[0], n_classes), device=device)
    oh[torch.arange(labels.shape[0]), labels.long()] = 1.0
    return oh


In [8]:
# =========================================
# 🧠 Conditional GAN (simple MLP)
# =========================================
class Generator(nn.Module):
    def __init__(self, noise_dim, n_classes, output_dim, hidden_dim=256):
        super().__init__()
        input_dim = noise_dim + n_classes
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, noise, labels_onehot):
        x = torch.cat([noise, labels_onehot], dim=1)
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self, input_dim, n_classes, hidden_dim=256):
        super().__init__()
        dim = input_dim + n_classes
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, features, labels_onehot):
        x = torch.cat([features, labels_onehot], dim=1)
        return self.net(x)


In [9]:
# =========================================
# 🔄 GAN training function
# =========================================
def train_conditional_gan(real_X, real_y, n_classes, device,
                          noise_dim=64, batch_size=1024, epochs=200,
                          lr=2e-4, gen_hidden=256, disc_hidden=256):
    input_dim = real_X.shape[1]
    G = Generator(noise_dim, n_classes, input_dim, gen_hidden).to(device)
    D = Discriminator(input_dim, n_classes, disc_hidden).to(device)

    optG = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    loss_fn = nn.BCELoss()

    dataset = TensorDataset(torch.tensor(real_X, dtype=torch.float32), torch.tensor(real_y, dtype=torch.long))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    for epoch in range(epochs):
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            bs = batch_X.size(0)

            real_labels = torch.ones((bs, 1), device=device)
            fake_labels = torch.zeros((bs, 1), device=device)

            # Train Discriminator
            D.zero_grad()
            y_onehot = to_onehot(batch_y, n_classes, device)
            pred_real = D(batch_X, y_onehot)
            loss_real = loss_fn(pred_real, real_labels)

            noise = torch.randn(bs, noise_dim, device=device)
            fake = G(noise, y_onehot).detach()
            pred_fake = D(fake, y_onehot)
            loss_fake = loss_fn(pred_fake, fake_labels)

            d_loss = (loss_real + loss_fake) / 2
            d_loss.backward()
            optD.step()

            # Train Generator
            G.zero_grad()
            noise = torch.randn(bs, noise_dim, device=device)
            gen = G(noise, y_onehot)
            pred_gen = D(gen, y_onehot)
            g_loss = loss_fn(pred_gen, real_labels)
            g_loss.backward()
            optG.step()

        if epoch % max(1, epochs // 10) == 0:
            print(f"[Epoch {epoch+1}/{epochs}] D_loss: {d_loss.item():.4f} | G_loss: {g_loss.item():.4f}")

    return G, D


In [10]:
# =========================================
# 🧮 Data loading + preprocessing
# =========================================
df = pd.read_csv(CSV_PATH)
print("Dataset shape:", df.shape)

# Drop text label if present
if "attack_cat_merged" in df.columns:
    df = df.drop(columns=["attack_cat_merged"])

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].values.astype(int)

# Convert non-numeric columns to codes
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].astype("category").cat.codes

# Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X.values, y, test_size=0.3, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

n_classes = int(np.max(y_train) + 1)
input_dim = X_train.shape[1]

print(f"✅ Classes: {n_classes}, Features: {input_dim}")
print("Train class distribution:", Counter(y_train))


Dataset shape: (257673, 41)
✅ Classes: 4, Features: 39
Train class distribution: Counter({2: 65100, 3: 42894, 1: 41210, 0: 31167})


In [11]:
# =========================================
# 🧬 Train the GAN
# =========================================
G, D = train_conditional_gan(
    real_X=X_train,
    real_y=y_train,
    n_classes=n_classes,
    device=DEVICE,
    noise_dim=NOISE_DIM,
    batch_size=GAN_BATCH_SIZE,
    epochs=GAN_EPOCHS,
    lr=GAN_LR
)


[Epoch 1/200] D_loss: 0.6054 | G_loss: 0.8121
[Epoch 21/200] D_loss: 0.4492 | G_loss: 1.3391
[Epoch 41/200] D_loss: 0.4265 | G_loss: 1.5298
[Epoch 61/200] D_loss: 0.3031 | G_loss: 1.8110
[Epoch 81/200] D_loss: 0.3148 | G_loss: 2.2951
[Epoch 101/200] D_loss: 0.2034 | G_loss: 3.1969
[Epoch 121/200] D_loss: 0.2117 | G_loss: 2.6366
[Epoch 141/200] D_loss: 0.1997 | G_loss: 2.7293
[Epoch 161/200] D_loss: 0.1538 | G_loss: 3.0672
[Epoch 181/200] D_loss: 0.2087 | G_loss: 1.9143


In [12]:
# =========================================
# 🧪 Generate synthetic samples to balance classes
# =========================================
def generate_synthetic_data(generator, target_counts, existing_counts, n_classes, noise_dim, device):
    Xs, ys = [], []
    for c in range(n_classes):
        need = target_counts.get(c, 0) - existing_counts.get(c, 0)
        if need <= 0:
            continue
        while need > 0:
            take = min(1024, need)
            z = torch.randn(take, noise_dim, device=device)
            labels = torch.tensor([c]*take, dtype=torch.long, device=device)
            onehot = to_onehot(labels, n_classes, device)
            with torch.no_grad():
                samples = generator(z, onehot).cpu().numpy()
            Xs.append(samples)
            ys.append(np.full(take, c))
            need -= take
    if len(Xs) == 0:
        return np.zeros((0, input_dim)), np.array([], dtype=int)
    return np.vstack(Xs), np.concatenate(ys)

existing_counts = Counter(y_train)
max_existing = max(existing_counts.values())
target_counts = {c: min(max_existing, MAX_SAMPLES_PER_CLASS) for c in range(n_classes)}

X_synth, y_synth = generate_synthetic_data(G, target_counts, existing_counts, n_classes, NOISE_DIM, DEVICE)
print("Synthetic samples generated:", X_synth.shape[0])

X_train_aug = np.vstack([X_train, X_synth])
y_train_aug = np.concatenate([y_train, y_synth])
print("Final train size:", X_train_aug.shape)


Synthetic samples generated: 0
Final train size: (180371, 39)


In [13]:
# =========================================
# 🚀 Train TabNet Classifier
# =========================================
tabnet = TabNetClassifier(
    n_d=64,
    n_a=64,
    n_steps=5,
    gamma=1.3,
    lambda_sparse=1e-3,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=TABNET_LR),
    mask_type='sparsemax'
)

tabnet.fit(
    X_train=X_train_aug, y_train=y_train_aug,
    eval_set=[(X_val, y_val)],
    eval_name=['val'],
    eval_metric=['accuracy'],
    max_epochs=TABNET_EPOCHS,
    patience=15,
    batch_size=TABNET_BATCH_SIZE,
    virtual_batch_size=TABNET_VBATCH_SIZE,
    num_workers=0,
    drop_last=False
)


/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.62    | val_accuracy: 0.79752 |  0:00:12s
epoch 1  | loss: 0.41999 | val_accuracy: 0.81214 |  0:00:23s
epoch 2  | loss: 0.39449 | val_accuracy: 0.80567 |  0:00:34s
epoch 3  | loss: 0.38244 | val_accuracy: 0.81928 |  0:00:45s
epoch 4  | loss: 0.37967 | val_accuracy: 0.81638 |  0:00:56s
epoch 5  | loss: 0.37407 | val_accuracy: 0.76277 |  0:01:08s
epoch 6  | loss: 0.36633 | val_accuracy: 0.80826 |  0:01:19s
epoch 7  | loss: 0.368   | val_accuracy: 0.81925 |  0:01:30s
epoch 8  | loss: 0.37636 | val_accuracy: 0.81706 |  0:01:41s
epoch 9  | loss: 0.37044 | val_accuracy: 0.75188 |  0:01:53s
epoch 10 | loss: 0.37947 | val_accuracy: 0.8117  |  0:02:04s
epoch 11 | loss: 0.3609  | val_accuracy: 0.7739  |  0:02:15s
epoch 12 | loss: 0.3577  | val_accuracy: 0.7775  |  0:02:26s
epoch 13 | loss: 0.35779 | val_accuracy: 0.80963 |  0:02:37s
epoch 14 | loss: 0.35624 | val_accuracy: 0.73142 |  0:02:48s
epoch 15 | loss: 0.36529 | val_accuracy: 0.78008 |  0:02:59s
epoch 16 | loss: 0.35549

/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [14]:
# =========================================
# 📊 Evaluate
# =========================================
preds = tabnet.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f" Test Accuracy: {acc:.4f}")
print(classification_report(y_test, preds, digits=4))

✅ Test Accuracy: 0.8279
              precision    recall  f1-score   support

           0     0.7889    0.5490    0.6475      6679
           1     0.9942    0.9685    0.9812      8831
           2     0.8798    0.9254    0.9020     13950
           3     0.6406    0.7477    0.6900      9191

    accuracy                         0.8279     38651
   macro avg     0.8259    0.7977    0.8052     38651
weighted avg     0.8334    0.8279    0.8257     38651



In [15]:
# =========================================
# 💾 Save models
# =========================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
tabnet.save_model(os.path.join(OUTPUT_DIR, "tabnet_model.zip"))
torch.save(G.state_dict(), os.path.join(OUTPUT_DIR, "gan_generator.pt"))
joblib.dump(scaler, os.path.join(OUTPUT_DIR, "scaler.joblib"))

print("✅ Models saved successfully in", OUTPUT_DIR)

Successfully saved model at ./models/tabnet_model.zip.zip
✅ Models saved successfully in ./models
